In [5]:
import pandas as pd
df = pd.read_csv('data_Histo/vnindex_eda_output.csv')
df.columns

Index(['symbol', 'date', 'high_price', 'low_price', 'open_price',
       'average_price', 'close_price', 'basic_price', 'adj_ratio', 'unit',
       'vol_total', 'vol_deal', 'vol_putth', 'val_total', 'val_putth',
       'buy_vol_foreign', 'buy_val_foreigh', 'sell_vol_foreign',
       'sel_val_foreign', 'buy_count', 'buy_vol', 'sell_count', 'sell_vol',
       'foreign_room', 'prop_trading_deal', 'prop_trading_putth',
       'prop_trading_net', 'year'],
      dtype='object')

In [6]:
df.head()

,symbol,date,high_price,low_price,open_price,average_price,close_price,basic_price,adj_ratio,unit,...,sel_val_foreign,buy_count,buy_vol,sell_count,sell_vol,foreign_room,prop_trading_deal,prop_trading_putth,prop_trading_net,year
0,VNINDEX,2010-01-04,517.05,501.74,501.74,517.05,517.05,494.77,1.0,1.0,...,1.863560e+11,35017.0,94121000.0,20679.0,47387200.0,0.0,NaN,NaN,NaN,2010
1,VNINDEX,2010-01-05,539.39,529.23,529.23,532.53,532.53,517.05,1.0,1.0,...,2.835830e+11,38919.0,114175000.0,34412.0,87391200.0,0.0,NaN,NaN,NaN,2010
2,VNINDEX,2010-01-06,538.84,526.37,529.47,534.46,534.46,532.53,1.0,1.0,...,1.578620e+11,44534.0,121988000.0,54391.0,113832000.0,0.0,NaN,NaN,NaN,2010
3,VNINDEX,2010-01-07,540.77,530.68,536.78,533.34,533.34,534.46,1.0,1.0,...,1.495040e+11,48977.0,127322000.0,46431.0,107237000.0,0.0,NaN,NaN,NaN,2010
4,VNINDEX,2010-01-08,544.49,520.90,540.95,520.90,520.90,533.34,1.0,1.0,...,1.995820e+11,47171.0,112076000.0,56106.0,139754000.0,0.0,NaN,NaN,NaN,2010


In [7]:
len(df)

3826

In [3]:
import pandas as pd
df = pd.read_parquet('data_News/equity_news_content_sentiment_ratios.parquet')
df.head(1)

,link,publication_date,domain_norm,category,title,description,keywords_norm,content,Tokenize_content_sentences,Tokenize_content,...,pos_term_count,neg_term_count,neutral_term_count,pos_ratio,neg_ratio,neutral_ratio,sentiment_coverage_ratio,polarity_count,sentiment_score,sentiment_label
0,https://vietstock.vn/2010/01/co-dong-noi-bo-ag...,2010-01-01 19:23:00,vietstock.vn,Giao dịch nội bộ,Cổ đông nội bộ AGF và BVH vi phạm CBTT,(Vietstock) - Sở GDCK TPHCM (HOSE) thông báo v...,<NA>,"Cụ thể, bà Nguyễn Thị Kim Lan đã mua 884,140 c...","[[cụ_thể, bà, nguyễn_thị_kim_lan, đã, mua, cổ_...","[cụ_thể, bà, nguyễn_thị_kim_lan, đã, mua, cổ_p...",...,2,0,8,0.060606,0.0,0.181818,0.242424,4,1.0,positive


In [11]:
len(df)


126576

In [4]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("data_News/equity_news_content_sentiment_ratios.parquet")
OUTPUT_PATH = Path("data_News/label_studio_ground_truth_stratified_200.csv")

RANDOM_STATE = 42

sample_plan = {
    "positive": 60,
    "negative": 60,
    "neutral": 60,
}

df = pd.read_parquet(INPUT_PATH).reset_index(names="source_row_id")

df["sentiment_label"] = (
    df["sentiment_label"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df = df.loc[
    df["content"].notna()
    & df["content"].astype(str).str.strip().ne("")
    & df["sentiment_label"].isin(sample_plan.keys())
].copy()

samples = []

for label, n in sample_plan.items():
    label_df = df.loc[df["sentiment_label"].eq(label)].copy()

    if len(label_df) < n:
        raise ValueError(f"Not enough rows for {label}: need {n}, got {len(label_df)}")

    samples.append(
        label_df.sample(n=n, random_state=RANDOM_STATE)
    )

sample_df = (
    pd.concat(samples, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

keep_columns = [
    "source_row_id",
    "publication_date",
    "domain_norm",
    "category",
    "title",
    "description",
    "content",
    "sentiment_score",
    "sentiment_label",
]

keep_columns = [col for col in keep_columns if col in sample_df.columns]
sample_df = sample_df[keep_columns].copy()

sample_df = sample_df.rename(columns={
    "sentiment_label": "model_sentiment_label",
    "sentiment_score": "model_sentiment_score",
})

sample_df.insert(0, "annotation_id", range(1, len(sample_df) + 1))

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sample_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_PATH)
print("Rows:", len(sample_df))
print(sample_df["model_sentiment_label"].value_counts())

Saved: data_News\label_studio_ground_truth_stratified_200.csv
Rows: 180
model_sentiment_label
positive    60
neutral     60
negative    60
Name: count, dtype: Int64
